# GSE221156 human pancreatic islets (stage 3): endothelial cells, genes of interest, ERG and pseudobulk

Reads the annotated object from stage 2 and does:

1. **Endothelial cells:** identify the cluster, show its markers (the dot plot panel of an adipose endothelial atlas, plus islet markers), re-cluster the endothelial cells with their own elbow plot and UMAP.
2. **Genes of interest** in endothelial cells, ND against T2D (PD as a separate comparison): ERG, PECAM1 (CD31), CDH5 (VE-cadherin), NOS3 (eNOS), VCAM1, ICAM1, HIF1A (HIF-1alpha), ANGPT2.
3. **ERG in every cell type:** the number and proportion of ERG-positive cells by cell type, chemistry, disease group and donor. **ERG-positive = at least 1 UMI in the original (uncorrected) counts**; zero means not detected, not absent.
4. **Donor-level pseudobulk** (counts summed per donor and cell type, edgeR): PD and T2D against ND, with age, BMI, sex and chemistry as covariates where the numbers allow; the same models **within V2 and within V3**; and a check of which models the design can support.

**Statistics rules used throughout.** Donors, not cells, are the replicates. Cells are summed per donor and cell type before any test. A donor with fewer than 20 cells of a type is left out for that type (missing, not zero).
Log2 fold changes are **disease relative to ND**. Adjusted p-values use Benjamini-Hochberg, within each analysis.

**Design limits (from stage 2).** V2 holds no lean donors and only one PD donor, so PD cannot be tested within V2. Lean donors are almost all in ND. Disease and chemistry are partly tied, and BMI differs between groups.
I tested the code on small made-up data only. Read each table before trusting it.

## 1. Setup

**Step:** Load the packages, set the options and read the annotated object.

In [ ]:
library(Seurat)
library(Matrix)
library(ggplot2)
library(edgeR)

out_dir <- "GSE221156_results"
analysis_assay <- "decontX"          # counts for clustering, and for pseudobulk (the original "RNA" is used as a sensitivity check)
min_cells_sample <- 20               # minimum cells of one type from one donor
min_donors_group <- 3                # donors needed in a group for a test
fdr_cut <- 0.05
n_variable_genes <- 2000
max_pcs <- 30

genes_of_interest <- c("ERG", "PECAM1", "CDH5", "NOS3", "VCAM1", "ICAM1", "HIF1A", "ANGPT2")
gene_labels <- c(ERG = "ERG", PECAM1 = "CD31 (PECAM1)", CDH5 = "VE-cadherin (CDH5)", NOS3 = "eNOS (NOS3)",
                 VCAM1 = "VCAM1", ICAM1 = "ICAM1", HIF1A = "HIF-1alpha (HIF1A)", ANGPT2 = "ANGPT2")

stage_dir <- file.path(out_dir, "stage3")
dir.create(stage_dir, showWarnings = FALSE, recursive = TRUE)
set.seed(1)

obj <- readRDS(file.path(out_dir, "GSE221156_annotated.rds"))
if (!analysis_assay %in% Assays(obj)) analysis_assay <- "RNA"
DefaultAssay(obj) <- analysis_assay
cat("assay used:", analysis_assay, "| genes of interest present:", sum(genes_of_interest %in% rownames(obj)), "of", length(genes_of_interest), "\n")
table(obj$cell_type)

## 2. The endothelial cluster

**Step:** Which cluster is endothelial? The cluster names come from stage 2; PECAM1 and CDH5 were not used to name them. Show the markers on a dot plot and feature plots.

In [ ]:
ec_label <- "Endothelial"
stopifnot(ec_label %in% obj$cell_type)

islet_ec_markers <- intersect(c("PLVAP", "VWF", "KDR", "EMCN", "ESAM", "FLT1", "RAMP2", "CLDN5", "ECSCR", "PECAM1", "CDH5"), rownames(obj))
p_ec_dot <- DotPlot(obj, features = islet_ec_markers, group.by = "cell_type", assay = analysis_assay) + RotatedAxis() +
  ggtitle("Endothelial markers across all cell types")
p_ec_dot
ggsave(file.path(stage_dir, "endothelial_markers_dotplot.png"), p_ec_dot, width = 10, height = 5, dpi = 140)

p_ec_feat <- FeaturePlot(obj, features = intersect(c("PLVAP", "VWF", "KDR", "PECAM1", "CDH5", "ERG"), rownames(obj)), ncol = 3, order = TRUE)
p_ec_feat
ggsave(file.path(stage_dir, "endothelial_markers_featureplot.png"), p_ec_feat, width = 14, height = 8, dpi = 110)

*What you should see:* The Endothelial population lights up for PLVAP, VWF, KDR and PECAM1/CDH5 and not for the other cell types. PLVAP is the endothelial marker the paper used.

**Step:** Subset the endothelial cells. Re-run the reduction on them: variable genes, scaling, PCA and the **elbow plot**.

In [ ]:
ec <- subset(obj, subset = cell_type == ec_label)
cat("endothelial cells:", ncol(ec), "from", length(unique(ec$donor)), "donors\n")

number_of_pcs <- function(stdev) {
  percent <- stdev / sum(stdev) * 100
  cumulative <- cumsum(percent)
  first  <- which(cumulative > 90 & percent < 5)[1]
  second <- sort(which((percent[-length(percent)] - percent[-1]) > 0.05), decreasing = TRUE)[1] + 1
  min(first, second, na.rm = TRUE)
}

ec$batch <- ec$donor
few <- names(which(table(ec$batch) < 30))
ec$batch[ec$batch %in% few] <- "small_donors"
ec[[analysis_assay]] <- split(ec[[analysis_assay]], f = ec$batch)
ec <- NormalizeData(ec)
ec <- FindVariableFeatures(ec, nfeatures = n_variable_genes)
ec <- ScaleData(ec)
ec <- RunPCA(ec, npcs = 30)

n_pcs_ec <- min(number_of_pcs(Stdev(ec, reduction = "pca")), max_pcs, 30)
elbow_ec <- ElbowPlot(ec, ndims = 30) + geom_vline(xintercept = n_pcs_ec, colour = "red", linetype = 2) +
  ggtitle(paste("Endothelial cells: elbow plot,", n_pcs_ec, "PCs used"))
elbow_ec
ggsave(file.path(stage_dir, "endothelial_elbow_plot.png"), elbow_ec, width = 6, height = 4, dpi = 150)

**Step:** Integrate the endothelial cells across donors, cluster them and make their UMAP.

In [ ]:
integrated <- NULL
k_weight <- 50
while (is.null(integrated) && k_weight >= 5) {
  integrated <- tryCatch(
    IntegrateLayers(ec, method = RPCAIntegration, orig.reduction = "pca", new.reduction = "integrated.rpca", k.weight = k_weight, verbose = FALSE),
    error = function(e) NULL)
  if (is.null(integrated)) k_weight <- floor(k_weight / 2)
}
if (is.null(integrated)) { reduction_ec <- "pca"; cat("integration failed: plain PCA used\n") } else { ec <- integrated; reduction_ec <- "integrated.rpca" }
ec[[analysis_assay]] <- JoinLayers(ec[[analysis_assay]])

ec <- FindNeighbors(ec, reduction = reduction_ec, dims = 1:n_pcs_ec)
ec <- FindClusters(ec, resolution = 0.5)
ec <- RunUMAP(ec, reduction = reduction_ec, dims = 1:n_pcs_ec)
table(ec$seurat_clusters)

### Endothelial sub-populations
The marker panel below is taken from an **adipose-tissue** endothelial atlas (venous, capillary, arterial, lymphatic, stromal and mural populations). **Islet capillaries are a different tissue and may not split the same way**, so treat the sub-populations as a first look:
a cluster that matches no panel stays `Unassigned_EC`. VCAM1, ICAM1, PECAM1 and CDH5 appear on the dot plot but are not used to name clusters, because they are genes you are testing.

**Step:** The panel genes, in the order of the atlas dot plot, and the groups used for naming.

In [ ]:
panel_order <- c("PECAM1", "CDH5", "ACKR1", "TLL1", "ELOVL7", "ICAM1", "VCAM1", "SELE", "SELP", "PLVAP", "CD74", "RBP7", "GPIHBP1",
                 "CADM2", "BTNL9", "CEACAM1", "RGCC", "CA4", "IFITM2", "IFI27", "GJA5", "NEBL", "LAMA2", "NEGR1", "CD44", "PDGFRA",
                 "PTPRC", "MRC1", "PROX1", "MMRN1", "INPP4B", "PDGFRB", "MYH11", "MYOCD")

ec_panels <- list(
  Venous      = c("ACKR1", "TLL1", "ELOVL7", "SELE", "SELP"),
  Capillary   = c("RBP7", "GPIHBP1", "CADM2", "BTNL9", "CEACAM1", "RGCC", "CA4"),
  Arterial    = c("GJA5", "NEBL"),
  Stromal     = c("LAMA2", "NEGR1", "CD44", "PDGFRA", "PTPRC", "MRC1"),
  Lymphatic   = c("PROX1", "MMRN1"),
  Mural       = c("INPP4B", "PDGFRB", "MYH11", "MYOCD")
)
ec_panels <- lapply(ec_panels, function(g) intersect(setdiff(g, genes_of_interest), rownames(ec)))
ec_panels <- ec_panels[lengths(ec_panels) > 0]
sapply(ec_panels, length)

**Step:** Name each endothelial cluster by the same rule as stage 2 (mean z-score against each panel, with a margin).

In [ ]:
avg <- AverageExpression(ec, assays = analysis_assay, features = unique(unlist(ec_panels)), group.by = "seurat_clusters", layer = "data")[[analysis_assay]]
avg <- log1p(as.matrix(avg))
colnames(avg) <- sub("^g", "", colnames(avg))
z <- t(scale(t(avg)))
z[is.na(z)] <- 0
scores <- sapply(ec_panels, function(genes) colMeans(z[genes, , drop = FALSE]))

ec_annotation <- do.call(rbind, lapply(rownames(scores), function(cl) {
  ranked <- sort(scores[cl, ], decreasing = TRUE)
  data.frame(cluster = cl, best = names(ranked)[1], score = round(ranked[1], 2), runner_up = names(ranked)[2],
             margin = round(ranked[1] - ranked[2], 2),
             label = ifelse(ranked[1] >= 0.25 & ranked[1] - ranked[2] >= 0.25, names(ranked)[1], "Unassigned_EC"))
}))
rownames(ec_annotation) <- NULL
ec$ec_type <- ec_annotation$label[match(as.character(ec$seurat_clusters), ec_annotation$cluster)]
write.csv(ec_annotation, file.path(stage_dir, "endothelial_cluster_annotation.csv"), row.names = FALSE)
ec_annotation
table(ec$ec_type)

*What you should see:* One row per endothelial cluster. In islets, most clusters may resolve to the capillary or venous panels; Unassigned_EC is a valid outcome.

**Step:** UMAP of the endothelial cells (clusters, sub-populations, chemistry, disease group), and the dot plot of the panel genes.

In [ ]:
p_a <- DimPlot(ec, group.by = "seurat_clusters", label = TRUE) + ggtitle("Endothelial clusters")
p_b <- DimPlot(ec, group.by = "ec_type", label = TRUE, repel = TRUE) + ggtitle("Endothelial sub-populations")
p_c <- DimPlot(ec, group.by = "chemistry") + ggtitle("Chemistry")
p_d <- DimPlot(ec, group.by = "group") + ggtitle("Disease group")
p_ec_umap <- (p_a + p_b) / (p_c + p_d)
p_ec_umap
ggsave(file.path(stage_dir, "endothelial_umap.png"), p_ec_umap, width = 13, height = 11, dpi = 120)

Idents(ec) <- "ec_type"
p_panel <- DotPlot(ec, features = intersect(panel_order, rownames(ec)), assay = analysis_assay, cols = c("lightgrey", "darkred")) +
  RotatedAxis() + ggtitle("Endothelial sub-population markers")
p_panel
ggsave(file.path(stage_dir, "endothelial_panel_dotplot.png"), p_panel, width = 16, height = 5, dpi = 130)

## 3. ERG in every cell type
**ERG-positive** means at least 1 UMI in the **original** counts (assay `RNA`, not the DecontX-corrected counts). A zero says ERG was not detected in that cell, not that it is absent. A stricter definition (at least 2 UMIs) is shown as a sensitivity check.
Percentages are of cells, so they depend on sequencing depth: compare like with like (the same cell type, chemistry and group).

**Step:** Mark each cell as ERG-positive or not, and count by cell type.

In [ ]:
raw <- LayerData(obj, assay = "RNA", layer = "counts")
obj$ERG_umis      <- as.numeric(raw["ERG", colnames(obj)])
obj$ERG_positive  <- obj$ERG_umis >= 1
obj$ERG_positive2 <- obj$ERG_umis >= 2
rm(raw)

erg_summary <- function(by) {
  groups <- split(obj@meta.data, by)
  do.call(rbind, lapply(names(groups), function(g) {
    d <- groups[[g]]
    data.frame(subgroup = g, cells = nrow(d), erg_positive = sum(d$ERG_positive), percent_positive = round(100 * mean(d$ERG_positive), 2),
               percent_2plus_umis = round(100 * mean(d$ERG_positive2), 2), donors = length(unique(d$donor)),
               donors_with_erg_cells = length(unique(d$donor[d$ERG_positive])))
  }))
}
erg_by_type <- erg_summary(obj$cell_type)
erg_by_type <- erg_by_type[order(-erg_by_type$percent_positive), ]
write.csv(erg_by_type, file.path(stage_dir, "ERG_by_cell_type.csv"), row.names = FALSE)
erg_by_type

*What you should see:* A row per cell type: cells, ERG-positive cells, percent positive, and how many donors contribute. Endothelial cells should have the highest percentage.

**Step:** ERG by cell type within each chemistry, disease group, and group x chemistry.

In [ ]:
erg_by_chemistry <- erg_summary(paste(obj$cell_type, obj$chemistry, sep = " | "))
erg_by_group     <- erg_summary(paste(obj$cell_type, obj$group, sep = " | "))
erg_by_both      <- erg_summary(paste(obj$cell_type, obj$group, obj$chemistry, sep = " | "))
write.csv(erg_by_chemistry, file.path(stage_dir, "ERG_by_cell_type_chemistry.csv"), row.names = FALSE)
write.csv(erg_by_group,     file.path(stage_dir, "ERG_by_cell_type_group.csv"), row.names = FALSE)
write.csv(erg_by_both,      file.path(stage_dir, "ERG_by_cell_type_group_chemistry.csv"), row.names = FALSE)
erg_by_group[grepl(ec_label, erg_by_group$subgroup), ]
erg_by_both[grepl(ec_label, erg_by_both$subgroup), ]

**Step:** ERG by donor within the endothelial cells, with the number of cells behind each percentage.

In [ ]:
is_ec <- obj$cell_type == ec_label
ec_meta <- obj@meta.data[is_ec, ]
erg_donor <- do.call(rbind, lapply(split(ec_meta, ec_meta$donor), function(d) {
  data.frame(donor = d$donor[1], group = d$group[1], chemistry = d$chemistry[1], ec_cells = nrow(d),
             erg_umis = sum(d$ERG_umis), erg_positive_cells = sum(d$ERG_positive), percent_positive = round(100 * mean(d$ERG_positive), 1))
}))
erg_donor <- erg_donor[order(erg_donor$chemistry, erg_donor$group, erg_donor$donor), ]
write.csv(erg_donor, file.path(stage_dir, "ERG_by_donor_endothelial.csv"), row.names = FALSE)
erg_donor

**Step:** Plots: percent ERG-positive cells by cell type, ERG expression by cell type, and ERG in endothelial cells by donor.

In [ ]:
p_pct <- ggplot(erg_by_type, aes(reorder(subgroup, percent_positive), percent_positive)) + geom_col(fill = "steelblue") + coord_flip() +
  geom_text(aes(label = paste0(percent_positive, "%")), hjust = -0.1) + scale_y_continuous(expand = expansion(mult = c(0, 0.15))) +
  labs(title = "ERG-positive cells by cell type (at least 1 UMI)", x = NULL, y = "% of cells") + theme_bw()
p_pct
ggsave(file.path(stage_dir, "ERG_percent_by_cell_type.png"), p_pct, width = 8, height = 5, dpi = 150)

p_vln <- VlnPlot(obj, features = "ERG", group.by = "cell_type", pt.size = 0, assay = analysis_assay) + NoLegend() + ggtitle("ERG expression (normalised) by cell type")
p_vln
ggsave(file.path(stage_dir, "ERG_violin_by_cell_type.png"), p_vln, width = 9, height = 5, dpi = 150)

p_donor <- ggplot(erg_donor, aes(donor, percent_positive, fill = group)) + geom_col() + facet_grid(. ~ chemistry, scales = "free_x", space = "free_x") +
  labs(title = "ERG-positive endothelial cells by donor", x = NULL, y = "% of the donor's endothelial cells") + theme_bw() +
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1))
p_donor
ggsave(file.path(stage_dir, "ERG_percent_endothelial_by_donor.png"), p_donor, width = 12, height = 5, dpi = 130)

## 4. Donor-level pseudobulk
Counts are **summed per donor and cell type**, so each donor is one replicate. edgeR (quasi-likelihood, TMM normalisation) tests PD and T2D against ND.
First the design is checked: which comparisons the donors can support, and which covariates are too closely tied to disease or chemistry to separate.

**Step:** Helper functions: sum counts per donor and cell type, and fit an edgeR model for one set of donors.

In [ ]:
pseudobulk <- function(object, cell_type_name, assay) {
  cells <- colnames(object)[object$cell_type == cell_type_name]
  counts <- LayerData(object, assay = assay, layer = "counts")[, cells, drop = FALSE]
  donor_of_cell <- factor(object$donor[match(cells, colnames(object))])
  to_donor <- sparseMatrix(i = seq_along(donor_of_cell), j = as.integer(donor_of_cell), x = 1,
                           dims = c(length(donor_of_cell), nlevels(donor_of_cell)), dimnames = list(NULL, levels(donor_of_cell)))
  sums <- as.matrix(counts %*% to_donor)
  per_donor <- unique(object@meta.data[, c("donor", "group", "chemistry", "sex", "age", "bmi")])
  info <- per_donor[match(colnames(sums), per_donor$donor), ]
  info$cells <- as.vector(table(donor_of_cell)[colnames(sums)])
  rownames(info) <- info$donor
  list(counts = sums, info = info)
}

fit_edger <- function(pb, formula, coefficient_pattern, force_genes = genes_of_interest) {
  info <- pb$info
  design <- model.matrix(formula, data = info)
  if (qr(design)$rank < ncol(design)) return(list(status = "design is not full rank", table = NULL))
  if (nrow(design) - ncol(design) < 1)  return(list(status = "no residual degrees of freedom", table = NULL))
  y <- DGEList(counts = pb$counts[, rownames(info)])
  keep <- filterByExpr(y, design = design) | rownames(y) %in% force_genes
  y <- y[keep, , keep.lib.sizes = FALSE]
  y <- tryCatch(normLibSizes(y), error = function(e) calcNormFactors(y))
  y <- estimateDisp(y, design, robust = TRUE)
  fit <- glmQLFit(y, design, robust = TRUE)
  results <- list()
  for (coefficient in grep(coefficient_pattern, colnames(design), value = TRUE)) {
    tt <- topTags(glmQLFTest(fit, coef = coefficient), n = Inf, adjust.method = "BH")$table
    tt$gene <- rownames(tt); tt$comparison <- coefficient
    results[[coefficient]] <- tt
  }
  list(status = "ok", table = do.call(rbind, results), y = y, design = design)
}

**Step:** Build the endothelial pseudobulk and the sample-size table: donors per group and chemistry with enough endothelial cells.

In [ ]:
pb_ec <- pseudobulk(obj, ec_label, analysis_assay)
pb_ec$info <- pb_ec$info[pb_ec$info$cells >= min_cells_sample, ]
pb_ec$counts <- pb_ec$counts[, rownames(pb_ec$info)]
pb_ec$info$group <- factor(pb_ec$info$group, levels = c("ND", "PD", "T2D"))     # ND is the reference
write.csv(pb_ec$info, file.path(stage_dir, "pseudobulk_endothelial_samples.csv"), row.names = FALSE)

cat("donors with >=", min_cells_sample, "endothelial cells:\n")
table(pb_ec$info$group, pb_ec$info$chemistry)
aggregate(cells ~ group + chemistry, data = pb_ec$info, FUN = sum)           # endothelial cells behind each cell of the design

*What you should see:* Donors per group and chemistry, and the endothelial cells behind them.

**Step:** **Which models can this design support?** For each candidate model: number of donors, columns, rank (full rank = estimable) and residual degrees of freedom. Interactions are tried the same way.

In [ ]:
info <- pb_ec$info
info$bmi_c <- as.numeric(scale(info$bmi, scale = FALSE))     # centred, so main effects stay interpretable with interactions
info$age_c <- as.numeric(scale(info$age, scale = FALSE))
info$chemistry <- factor(info$chemistry)

candidates <- list(
  "group"                              = ~ group,
  "chemistry + group"                  = ~ chemistry + group,
  "group + age + bmi"                  = ~ group + age_c + bmi_c,
  "chemistry + group + age + bmi"      = ~ chemistry + group + age_c + bmi_c,
  "chemistry + group + age + bmi + sex" = ~ chemistry + group + age_c + bmi_c + sex,
  "group x chemistry"                  = ~ group * chemistry,
  "group x bmi"                        = ~ group * bmi_c,
  "group x age"                        = ~ group * age_c,
  "chemistry x bmi"                    = ~ chemistry * bmi_c
)
design_check <- do.call(rbind, lapply(names(candidates), function(name) {
  d <- model.matrix(candidates[[name]], data = info)
  rank <- qr(d)$rank
  data.frame(model = name, donors = nrow(d), columns = ncol(d), rank = rank, residual_df = nrow(d) - rank,
             estimable = rank == ncol(d) && nrow(d) - rank >= 2)
}))
write.csv(design_check, file.path(stage_dir, "design_check.csv"), row.names = FALSE)
design_check

*What you should see:* estimable = TRUE when the model has full rank and at least 2 residual degrees of freedom. Models with interactions or many covariates often fail here with so few donors.

**Step:** How tied together are the variables? Disease against chemistry, BMI against disease, and age against disease: the closer the tie, the less a model can separate them.

In [ ]:
cat("Donors by group and chemistry:\n"); print(table(info$group, info$chemistry))

kruskal_p <- function(x, g) signif(kruskal.test(x ~ g)$p.value, 3)
cat("\nBMI differs between groups (Kruskal p):", kruskal_p(info$bmi, info$group), "\n")
cat("Age differs between groups (Kruskal p):", kruskal_p(info$age, info$group), "\n")
cat("BMI differs between chemistries (Wilcoxon p):", signif(wilcox.test(bmi ~ chemistry, data = info, exact = FALSE)$p.value, 3), "\n")
cat("Group and chemistry independent? (Fisher p):", signif(fisher.test(table(info$group, info$chemistry))$p.value, 3), "\n")
cat("\nMedian BMI and age by group:\n"); print(aggregate(cbind(bmi, age) ~ group, data = info, FUN = median))

*What you should see:* Small p-values mean the variables are not independent, so an adjusted estimate depends on the model. Report which covariates were used.

**Step:** Run the **combined** model: group (PD and T2D against ND) with chemistry, age and BMI as covariates where the design allows. The most adjusted model that is estimable is used.

In [ ]:
formula_order <- list(~ chemistry + group + age_c + bmi_c + sex, ~ chemistry + group + age_c + bmi_c, ~ chemistry + group + age_c, ~ chemistry + group, ~ group)
fit_combined <- NULL
for (f in formula_order) {
  check <- fit_edger(list(counts = pb_ec$counts, info = info), f, "^group")
  if (check$status == "ok") { fit_combined <- check; used_formula <- f; break }
}
cat("model used:", deparse(used_formula), "\n")
combined <- fit_combined$table
combined$donors_ND  <- sum(info$group == "ND")
combined$donors_PD  <- sum(info$group == "PD")
combined$donors_T2D <- sum(info$group == "T2D")
write.csv(combined, file.path(stage_dir, "pseudobulk_endothelial_combined_allgenes.csv"), row.names = FALSE)
combined_panel <- combined[combined$gene %in% genes_of_interest, ]
combined_panel$FDR_panel <- ave(combined_panel$PValue, combined_panel$comparison, FUN = function(p) p.adjust(p, "BH"))
combined_panel[, c("gene", "comparison", "logFC", "PValue", "FDR", "FDR_panel", "donors_ND", "donors_PD", "donors_T2D")]

*What you should see:* For each of groupPD and groupT2D (against ND): log2 fold change, p-value, genome-wide FDR and FDR across the eight genes, and the donors behind them.

**Step:** Run the same models **within V2** and **within V3**, where a single chemistry removes the chemistry effect. V2 has one PD donor, so only T2D can be tested there.

In [ ]:
within <- list()
for (chem in c("V2", "V3")) {
  sub_info <- info[info$chemistry == chem, ]
  sub_info$group <- droplevels(sub_info$group)
  n_by_group <- table(sub_info$group)
  cat("\n", chem, ": donors by group:", paste(names(n_by_group), n_by_group, collapse = ", "), "\n")
  usable <- names(n_by_group)[n_by_group >= min_donors_group]
  if (!all(c("ND", "T2D") %in% usable) && !all(c("ND", "PD") %in% usable)) { cat("  not enough donors for a group test\n"); next }
  sub_info <- sub_info[sub_info$group %in% usable, ]
  sub_info$group <- droplevels(factor(sub_info$group, levels = c("ND", setdiff(usable, "ND"))))
  fit <- NULL
  for (f in list(~ group + age_c + bmi_c + sex, ~ group + age_c + bmi_c, ~ group + age_c, ~ group)) {
    attempt <- fit_edger(list(counts = pb_ec$counts[, rownames(sub_info)], info = sub_info), f, "^group")
    if (attempt$status == "ok") { fit <- attempt; cat("  model used:", deparse(f), "\n"); break }
  }
  if (is.null(fit)) { cat("  no estimable model\n"); next }
  tab <- fit$table; tab$chemistry <- chem
  within[[chem]] <- tab
}
within_all <- do.call(rbind, within)
write.csv(within_all, file.path(stage_dir, "pseudobulk_endothelial_within_chemistry_allgenes.csv"), row.names = FALSE)
if (!is.null(within_all)) within_all[within_all$gene %in% genes_of_interest, c("chemistry", "gene", "comparison", "logFC", "PValue", "FDR")]

*What you should see:* V2: groupT2D only (PD has one donor). V3: groupPD and groupT2D. The covariates used are printed for each chemistry.

**Step:** Do V2 and V3 agree? Compare the T2D-versus-ND results of the eight genes across the within-chemistry and combined analyses, and count genes found in each.

In [ ]:
pick <- function(tab, label) {
  t <- tab[tab$comparison == "groupT2D" & tab$gene %in% genes_of_interest, c("gene", "logFC", "PValue")]
  names(t) <- c("gene", paste0("logFC_", label), paste0("p_", label)); t
}
agree <- Reduce(function(a, b) merge(a, b, by = "gene", all = TRUE),
                list(pick(combined, "combined"),
                     if (!is.null(within_all)) pick(within_all[within_all$chemistry == "V2", ], "V2") else data.frame(gene = genes_of_interest),
                     if (!is.null(within_all)) pick(within_all[within_all$chemistry == "V3", ], "V3") else data.frame(gene = genes_of_interest)))
agree$same_direction_V2_V3 <- sign(agree$logFC_V2) == sign(agree$logFC_V3)
write.csv(agree, file.path(stage_dir, "agreement_V2_V3_combined_T2D.csv"), row.names = FALSE)
agree

if (!is.null(within_all)) {
  significant <- function(chem) within_all$gene[within_all$chemistry == chem & within_all$comparison == "groupT2D" & within_all$FDR < fdr_cut]
  sig_V2 <- significant("V2"); sig_V3 <- significant("V3"); sig_all <- combined$gene[combined$comparison == "groupT2D" & combined$FDR < fdr_cut]
  cat("\nGenes with FDR <", fdr_cut, "(T2D vs ND): V2 =", length(sig_V2), "| V3 =", length(sig_V3), "| combined =", length(sig_all),
      "| in both V2 and V3 =", length(intersect(sig_V2, sig_V3)), "\n")
}

*What you should see:* A row per gene with log2 fold change in each analysis and whether V2 and V3 point the same way; then the number of significant genes in each and in both.

**Step:** **Sensitivity:** repeat the combined model on the **original** counts (assay RNA, no ambient correction) for the eight genes and compare the fold changes.

In [ ]:
sens <- NULL
if (analysis_assay != "RNA") {
  pb_raw <- pseudobulk(obj, ec_label, "RNA")
  pb_raw$info <- pb_raw$info[rownames(pb_raw$info) %in% rownames(info), ]
  pb_raw$counts <- pb_raw$counts[, rownames(pb_raw$info)]
  pb_raw$info <- cbind(pb_raw$info, info[rownames(pb_raw$info), c("age_c", "bmi_c")])
  pb_raw$info$group <- factor(pb_raw$info$group, levels = c("ND", "PD", "T2D"))
  raw_fit <- fit_edger(pb_raw, used_formula, "^group")
  if (raw_fit$status == "ok") {
    r <- raw_fit$table[raw_fit$table$gene %in% genes_of_interest & raw_fit$table$comparison == "groupT2D", c("gene", "logFC", "PValue")]
    names(r) <- c("gene", "logFC_original_counts", "p_original_counts")
    sens <- merge(r, combined[combined$comparison == "groupT2D" & combined$gene %in% genes_of_interest, c("gene", "logFC", "PValue")], by = "gene")
    names(sens)[4:5] <- c("logFC_decontX", "p_decontX")
  }
}
sens

*What you should see:* Fold changes from original and DecontX-corrected counts should agree in direction and size. A gene that changes sign is not robust to ambient RNA.

**Step:** **Interactions.** Fit the interaction models the design can support (disease x chemistry, disease x BMI, disease x age, chemistry x BMI) and report the interaction terms for the eight genes. Models the donors cannot support are named and skipped.

In [ ]:
interaction_models <- list(
  "group x chemistry" = ~ group * chemistry,
  "group x bmi"       = ~ group * bmi_c,
  "group x age"       = ~ group * age_c,
  "chemistry x bmi"   = ~ chemistry * bmi_c
)
interaction_results <- list()
for (name in names(interaction_models)) {
  estimable <- design_check$estimable[design_check$model == name]
  if (!isTRUE(estimable)) { cat(name, ": cannot be estimated with these donors, skipped\n"); next }
  fit <- fit_edger(list(counts = pb_ec$counts, info = info), interaction_models[[name]], ":")
  if (fit$status != "ok") { cat(name, ":", fit$status, ", skipped\n"); next }
  tab <- fit$table
  tab$model <- name
  interaction_results[[name]] <- tab
  cat(name, ": genes with interaction FDR <", fdr_cut, "=", sum(tab$FDR < fdr_cut), "\n")
}
interaction_all <- do.call(rbind, interaction_results)
if (!is.null(interaction_all)) {
  write.csv(interaction_all, file.path(stage_dir, "pseudobulk_endothelial_interactions_allgenes.csv"), row.names = FALSE)
  interaction_all[interaction_all$gene %in% genes_of_interest, c("model", "comparison", "gene", "logFC", "PValue", "FDR")]
}

*What you should see:* A row per interaction term and gene. A non-significant interaction does not show that there is none: with this few donors these tests have little power, and they mainly tell you whether the effect of disease looks different between chemistries, BMI levels or ages.

**Step:** **Lean and not lean.** Disease cannot be compared within lean donors (no PD donor and at most one T2D donor is lean). What the data do allow is lean against not-lean **within ND, V3 only**: a BMI question, not a diabetes one. Exploratory.

In [ ]:
nd_v3 <- info[info$group == "ND" & info$chemistry == "V3", ]
nd_v3$lean <- factor(ifelse(nd_v3$bmi < 25, "Lean", "Not lean"), levels = c("Not lean", "Lean"))
cat("ND donors in V3 with enough endothelial cells:", paste(names(table(nd_v3$lean)), table(nd_v3$lean), collapse = ", "), "\n")

lean_fit <- NULL
if (all(table(nd_v3$lean) >= min_donors_group)) {
  lean_fit <- fit_edger(list(counts = pb_ec$counts[, rownames(nd_v3)], info = nd_v3), ~ lean + age_c + sex, "^lean")
  if (lean_fit$status == "ok") {
    lean_tab <- lean_fit$table[lean_fit$table$gene %in% genes_of_interest, c("gene", "comparison", "logFC", "PValue", "FDR")]
    write.csv(lean_fit$table, file.path(stage_dir, "pseudobulk_endothelial_lean_vs_notlean_ND_V3_allgenes.csv"), row.names = FALSE)
    print(lean_tab)
  } else cat(lean_fit$status, "\n")
} else cat("fewer than", min_donors_group, "donors in one class: not tested\n")

*What you should see:* Donors per class, then log2 fold changes (Lean relative to Not lean) for the eight genes. Treat as exploratory: the groups differ in many things besides BMI.

### The eight genes: effect sizes, adjusted p-values and donors

**Step:** One summary table for the eight genes in endothelial cells: group comparison, effect size, p-value, adjusted p-value across the eight genes, and the number of donors in each group.

In [ ]:
summary_genes <- combined_panel
summary_genes$gene_label <- gene_labels[summary_genes$gene]
summary_genes$call <- ifelse(summary_genes$FDR_panel < fdr_cut, "significant (FDR < 0.05)", "not significant")
summary_genes$note <- ifelse(summary_genes$comparison == "groupPD" & summary_genes$donors_PD < min_donors_group, "fewer than 3 PD donors: interpret with great caution",
                      ifelse(summary_genes$comparison == "groupT2D" & summary_genes$donors_T2D < min_donors_group, "fewer than 3 T2D donors", ""))
reads <- rowSums(pb_ec$counts[summary_genes$gene, , drop = FALSE])
summary_genes$total_reads <- reads
summary_genes$note[summary_genes$total_reads < 50 & summary_genes$note == ""] <- "very few reads: insufficient information"
out <- summary_genes[, c("gene_label", "comparison", "logFC", "PValue", "FDR_panel", "donors_ND", "donors_PD", "donors_T2D", "total_reads", "call", "note")]
out <- out[order(out$comparison, out$PValue), ]
write.csv(out, file.path(stage_dir, "genes_of_interest_endothelial_summary.csv"), row.names = FALSE)
out

*What you should see:* ERG and the other genes with log2 fold change (disease vs ND), p-value, FDR across the eight genes, donors per group and a note when the evidence is thin.

**Step:** Donor-level plots: expression of the eight genes in endothelial cells by disease group, one dot per donor, split by chemistry.

In [ ]:
y <- fit_combined$y
log_cpm <- cpm(y, log = TRUE, prior.count = 1)
genes_present <- intersect(genes_of_interest, rownames(log_cpm))
plot_data <- do.call(rbind, lapply(genes_present, function(g) {
  data.frame(gene = gene_labels[g], donor = colnames(log_cpm), logCPM = log_cpm[g, ], info[colnames(log_cpm), c("group", "chemistry")])
}))
plot_data$gene <- factor(plot_data$gene, levels = gene_labels[genes_present])
p_genes <- ggplot(plot_data, aes(group, logCPM, fill = group)) + geom_boxplot(outlier.shape = NA, alpha = 0.6) +
  geom_jitter(aes(shape = chemistry), width = 0.12, size = 2) + facet_wrap(~ gene, scales = "free_y", ncol = 4) +
  labs(title = "Endothelial pseudobulk expression by disease group (one dot = one donor)", x = NULL, y = "log2 CPM (TMM-normalised)") +
  theme_bw() + theme(legend.position = "bottom")
p_genes
ggsave(file.path(stage_dir, "genes_of_interest_by_group.png"), p_genes, width = 13, height = 7, dpi = 130)

**Step:** Volcano plots of all endothelial genes for each comparison, with the eight genes labelled.

In [ ]:
volcano <- combined
volcano$significant <- volcano$FDR < fdr_cut
volcano$label <- ifelse(volcano$gene %in% genes_of_interest, volcano$gene, "")
p_volcano <- ggplot(volcano, aes(logFC, -log10(PValue), colour = significant)) + geom_point(alpha = 0.5, size = 1) +
  geom_text(aes(label = label), colour = "black", size = 3, vjust = -0.6, check_overlap = TRUE) +
  facet_wrap(~ comparison) + scale_colour_manual(values = c("grey60", "firebrick")) +
  labs(title = "Endothelial cells: differential expression against ND (donor-level)", x = "log2 fold change", y = "-log10 p-value") + theme_bw()
p_volcano
ggsave(file.path(stage_dir, "endothelial_volcano.png"), p_volcano, width = 11, height = 5, dpi = 130)

**Step:** Summary of endothelial cells and donors behind each comparison: cells per donor by group and chemistry.

In [ ]:
p_cells <- ggplot(pb_ec$info, aes(donor, cells, fill = group)) + geom_col() + facet_grid(. ~ chemistry, scales = "free_x", space = "free_x") +
  geom_hline(yintercept = min_cells_sample, linetype = 2) +
  labs(title = "Endothelial cells per donor (dashed = minimum used)", x = NULL, y = "endothelial cells") + theme_bw() +
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1))
p_cells
ggsave(file.path(stage_dir, "endothelial_cells_per_donor.png"), p_cells, width = 12, height = 5, dpi = 130)

### All cell types

**Step:** Run the pseudobulk comparison for every cell type with enough donors, and summarise: donors per group, number of genes with FDR below 0.05, and the results for the eight genes.

In [ ]:
types <- names(which(table(obj$cell_type) > 0))
types <- setdiff(types, "Unassigned")
all_types <- list()
type_summary <- list()
for (tp in types) {
  pb <- pseudobulk(obj, tp, analysis_assay)
  pb$info <- pb$info[pb$info$cells >= min_cells_sample, ]
  n <- table(factor(pb$info$group, levels = c("ND", "PD", "T2D")))
  row <- data.frame(cell_type = tp, donors_ND = n[["ND"]], donors_PD = n[["PD"]], donors_T2D = n[["T2D"]], tested = FALSE, genes_FDR_T2D = NA, genes_FDR_PD = NA)
  if (n[["ND"]] >= min_donors_group && n[["T2D"]] >= min_donors_group) {
    pb$counts <- pb$counts[, rownames(pb$info)]
    pb$info$group <- factor(pb$info$group, levels = c("ND", "PD", "T2D"))
    pb$info$age_c <- as.numeric(scale(pb$info$age, scale = FALSE)); pb$info$bmi_c <- as.numeric(scale(pb$info$bmi, scale = FALSE))
    pb$info$chemistry <- factor(pb$info$chemistry)
    if (n[["PD"]] < min_donors_group) { pb$info <- pb$info[pb$info$group != "PD", ]; pb$info$group <- droplevels(pb$info$group); pb$counts <- pb$counts[, rownames(pb$info)] }
    fit <- NULL
    for (f in list(~ chemistry + group + age_c + bmi_c, ~ chemistry + group, ~ group)) {
      attempt <- fit_edger(pb, f, "^group")
      if (attempt$status == "ok") { fit <- attempt; break }
    }
    if (!is.null(fit)) {
      tab <- fit$table; tab$cell_type <- tp; all_types[[tp]] <- tab
      row$tested <- TRUE
      row$genes_FDR_T2D <- sum(tab$comparison == "groupT2D" & tab$FDR < fdr_cut)
      row$genes_FDR_PD  <- sum(tab$comparison == "groupPD"  & tab$FDR < fdr_cut)
    }
  }
  type_summary[[tp]] <- row
}
type_summary <- do.call(rbind, type_summary)
write.csv(type_summary, file.path(stage_dir, "pseudobulk_all_cell_types_summary.csv"), row.names = FALSE)
type_summary

*What you should see:* A row per cell type: donors with enough cells in each group, whether it could be tested, and how many genes passed FDR < 0.05 for T2D and PD against ND.

**Step:** The eight genes across all tested cell types: log2 fold change (T2D vs ND) as a heat map, with the endothelial row for reference.

In [ ]:
if (length(all_types) > 0) {
  panel_rows <- do.call(rbind, lapply(all_types, function(tab) tab[tab$gene %in% genes_of_interest & tab$comparison == "groupT2D", c("cell_type", "gene", "logFC", "PValue", "FDR")]))
  write.csv(panel_rows, file.path(stage_dir, "genes_of_interest_all_cell_types_T2D.csv"), row.names = FALSE)
  panel_rows$gene <- factor(panel_rows$gene, levels = genes_of_interest)
  panel_rows$star <- ifelse(panel_rows$FDR < fdr_cut, "*", "")
  p_heat <- ggplot(panel_rows, aes(gene, cell_type, fill = logFC)) + geom_tile(colour = "white") + geom_text(aes(label = star), size = 6) +
    scale_fill_gradient2(low = "steelblue", mid = "white", high = "firebrick", midpoint = 0) +
    labs(title = "T2D vs ND: log2 fold change of the eight genes (* FDR < 0.05 genome-wide)", x = NULL, y = NULL) + theme_minimal()
  p_heat
  ggsave(file.path(stage_dir, "genes_of_interest_all_cell_types_heatmap.png"), p_heat, width = 9, height = 5, dpi = 140)
}

## Methods and limits
**Cells and groups.** Quality-controlled islet cells from stage 1 (cut-offs and sample sizes in the stage 1 exclusion log), integrated across donors with Seurat RPCA, clustered and named with marker panels written from memory of the Human Protein Atlas; check them against the atlas.
**ERG-positive** = at least 1 UMI in the original counts. **Pseudobulk** = counts summed per donor and cell type, at least 20 cells per donor and type, edgeR quasi-likelihood with TMM normalisation; genes of interest are always kept in the test.
**Covariates** used are printed in each model; the notebook uses the most adjusted model the design can estimate and reports when a model or an interaction cannot be fitted.

**Be careful.**
* Disease and chemistry are partly tied; BMI differs between groups; V2 has no lean donors and one PD donor. A combined model cannot fully separate these, so report the within-V2 and within-V3 results next to the combined one.
* Lean-versus-not-lean disease comparisons are not possible: the only lean T2D donor (Islet84) is recovered only with a barcode table. Lean ND against not-lean ND donors (all V3) is possible but is a BMI question, not a diabetes one.
* The endothelial sub-population panel comes from adipose tissue. Treat sub-populations in islets as exploratory.
* This is pancreatic islets, not adipose tissue; an ERG result here says nothing about fat.